In [1]:
#main file
import os
import math
import time
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
from tqdm.notebook import tqdm
import numpy as np

from torch.utils.data import DataLoader
from torchvision.utils import save_image
from torchvision.io import read_image
from torchvision.transforms import Resize
from torchvision.transforms import Compose, ToTensor, Lambda, Pad
from PIL import Image
nn = torch.nn
F = nn.functional

data = np.load('train_set_mini/annotations/7_aro.npy')
print(data)

data = np.load('train_set_mini/annotations/7_exp.npy')
print(data)

#data = np.load('train_set_mini/annotations/7_lnd.npy')
#print(data)

data = np.load('train_set_mini/annotations/7_val.npy')
print(data)

print(torch.cuda.is_available())

0.831839
6
-0.488145
True


In [35]:
class EmotionDetectionDataset(torch.utils.data.Dataset):
    def __init__(self, num_samples, train=True): 
        self.file_names = []
        self.samples = []
        self.arousals = []
        self.emotions = []
        self.valences = []
        #self.data = None
        self.train = train
        split = 'train' if train else 'test'

        if self.train:
            path = 'train_set_mini/annotations/'
        else:
            path = 'val_set/annotations/'
        #LOAD ALL LABELS
        for i in range(num_samples):
            #if i % 100 == 0:
                #print(i)
            try:
                file_name = path + str(i) + '_aro.npy'
                arousal = np.load(file_name)
                self.arousals.append(arousal)
                
                file_name = path + str(i) + '_val.npy'
                val = np.load(file_name)
                self.valences.append(val)
                
                file_name = path + str(i) + '_exp.npy'
                exp = np.load(file_name)
                self.emotions.append(exp)
        
                self.file_names.append(i)
                #self.get_plottable(i)
                if self.train:
                    self.samples.append(self.get_plottable('train_set_mini/images/' + str(i) + '.jpg'))
                else:
                    self.samples.append(self.get_plottable('val_set/images/' + str(i) + '.jpg'))
                #self.samples.append(self.get_plottable(path + str(i) +)
            except:
                continue


       # 
      
    def __len__(self): 
        return len(self.file_names)

    def __getitem__(self, idx): 
        name = self.file_names[idx]
        sample = (read_image(f'train_set_mini/images/{name}.jpg')/255.)
        return sample 

    def get_plottable(self, file_name): 
        #name = self.file_names[idx]
        sample = read_image(file_name).permute(1,2,0)#f'train_set_mini/images/{name}.jpg').permute(1,2,0) 
        return sample 



In [33]:
#plt.imshow(Set.get_plottable(0))

class OurDataset(Dataset):
    def __init__(self, samples, exp, val, aro):
        self.samples = samples
        self.exp = exp
        self.aro = aro
        self.val = val
    
    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = torch.tensor(np.copy(self.samples[idx])).to(torch.float32)#.astype(np.float32)))#.clone().detach()
        exp, val, aro = torch.tensor(np.copy(int(self.exp[idx]))), torch.tensor(np.copy(self.val[idx].astype(np.float32))), torch.tensor(np.copy(self.aro[idx].astype(np.float32)))
        return sample, exp, val, aro

In [4]:
class FaceReader(nn.Module):
    def __init__(self):
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        super().__init__()
        self.model = torch.nn.Sequential(torch.nn.Conv2d(3,50, kernel_size = 3, padding = 1, bias = False),
                                         torch.nn.BatchNorm2d(50),
                                         torch.nn.AvgPool2d(kernel_size = 5),
                                         #torch.nn.Conv2d(50,50, kernel_size = 3, padding = 1, bias = False),
                                         #torch.nn.BatchNorm2d(50),
                                         #torch.nn.AvgPool2d(kernel_size = 3),
                                         torch.nn.Conv2d(50,8, kernel_size = 3, padding = 1, bias = False),
                                         torch.nn.BatchNorm2d(8),
                                         torch.nn.AvgPool2d(kernel_size = 5),
                                         torch.nn.Flatten(),
                                         torch.nn.Linear(512,100),
                                        # torch.nn.ReLU(),
                                         torch.nn.Linear(100,8))
                                        
    def forward(self, x):
        y = self.model(x)
        return y

In [40]:
class EmotionTrainer():
    def __init__(self, model, batch_size,  opt, lr, Dataset1, Dataset2):
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        print('CUDA: ' + str(torch.cuda.is_available()))
        self.model = model()
        self.model.to(device = self.device, dtype = torch.float)
        self.train_data = DataLoader(Dataset1, batch_size = 32, shuffle = True)
        self.test_data = DataLoader(Dataset2, batch_size = 32, shuffle = False)
        self.optimizer = opt(self.model.parameters(), lr)
        self.loss = torch.nn.MSELoss()
        self.batch_size = batch_size
        
    def train(self, epochs):
        loss_history = torch.zeros((5))
        lh = []
        start = time.time()
        self.model.train()
        for epoch in range(epochs):
            print('Epoch: ', epoch+1)
            pbar = tqdm(self.train_data)
            acc_loss = 0
            acc_acc = 0
            count = 0
            for e,(sample, exp, val, aro) in enumerate(pbar,1): 
                self.optimizer.zero_grad()
                sample, exp, val, aro = sample.to(self.device), exp.to(self.device), val.to(self.device), aro.to(self.device)
                sample = sample.permute(0,3,1,2)#1,2,0)
                pred = self.model(sample)
                #print(pred)
                exp = F.one_hot(exp.long(), num_classes=8).to(dtype=torch.float)
                #print(exp.shape)
                #print(pred[0])
                loss = F.cross_entropy(pred,exp)
                count += 1
                #print(loss)
                loss.backward()
                self.optimizer.step()
                #print(e)
                acc_loss += loss.float()
                #print("pred: ", pred)
                correct = 0
                for i in range(len(pred)):
                    #print("guess: ", torch.argmax(pred[i]))
                    if torch.argmax(pred[i])==torch.argmax(exp[i]):
                        correct += 1
                #guesstimation = torch.argmax(pred)
                #print("guess: ", argmax(pred[i]))
                #correct = torch.sum(guesstimation == torch.argmax(exp))
                accuracy = correct/self.batch_size
                #print("batch accuracy: ", accuracy)
                acc_acc += accuracy
                #if guesstimation == torch.argmax(exp):
                #n = min(e,5)
                # .item() takes the single tensor value and makes it a python native element
                #loss_history[e] = loss.item() 
                #lh.append(loss.item())
                # this line will print the rolling loss average on the progress bar
            lh.append(acc_loss / count)
            print("Loss: ", acc_loss.item()/count)
            acc_acc = acc_acc / count
            print("Acc for epoch: ", acc_acc)

            #pbar.set_postfix(lh)

            #pbar.set_postfix(Loss = (loss_history.sum()/n).item())
        
        print('Total Training Time: ', time.time() - start)
        return lh
        
    @torch.inference_mode()
    def test(self):
        # This function calculates the mean average distance of the pixels 
        ### and subtracts it from 1 to get an accuracy
        def noiseAccuracy(predicted, target):
            size = target.flatten().shape[0]
            true = (predicted - target).abs().mean()
            return 1-true
        self.model.train(False)
        total_accuracy = 0
        start = time.time()
        pbar = tqdm(self.test_data)
        total_steps = 0
        acc_loss = 0
        acc_acc = 0
        for e,(sample, exp, val, aro) in enumerate(pbar,1): 
            sample, exp, val, aro = sample.to(self.device), exp.to(self.device), val.to(self.device), aro.to(self.device)
            sample = sample.permute(0,3,1,2)
            pred = self.model(sample)

            exp = F.one_hot(exp.long(), num_classes=8).to(dtype=torch.float)
            loss = F.cross_entropy(pred,exp)
            

            acc_loss += loss.float()
            #print("pred: ", pred)
            correct = 0
            for i in range(len(pred)):
                #print("guess: ", torch.argmax(pred[i]))
                if torch.argmax(pred[i])==torch.argmax(exp[i]):
                    correct += 1
            #guesstimation = torch.argmax(pred)
            #print("guess: ", argmax(pred[i]))
            #correct = torch.sum(guesstimation == torch.argmax(exp))
            accuracy = correct/self.batch_size
            #print("batch accuracy: ", accuracy)
            acc_acc += accuracy
            #lh.append(acc_loss / 87)
        print("Test loss: ", acc_loss/87)
        acc_acc = acc_acc / 87
        print("Test Acc: ", acc_acc)

In [46]:
Set1 = EmotionDetectionDataset(40000)
#print(Set.samples[4].shape)
Set2 = EmotionDetectionDataset(train=False, num_samples=8000)
print(Set1.__len__())
print(Set2.__len__())




13888
3999


In [ ]:
Dataset1 = OurDataset(Set1.samples, Set1.emotions, Set1.valences, Set1.arousals)
Dataset2 = OurDataset(Set2.samples, Set2.emotions, Set2.valences, Set2.arousals)
trainer = EmotionTrainer(FaceReader, 128, torch.optim.RAdam, 0.05, Dataset1, Dataset2)
t = trainer.train(30)


CUDA: True
Epoch:  1


  0%|          | 0/434 [00:00<?, ?it/s]

In [45]:
v = trainer.test()

  0%|          | 0/92 [00:00<?, ?it/s]

Test loss:  tensor(3.2577, device='cuda:0')
Test Acc:  0.21623563218390804
